In [7]:
MODEL_PATH = "/home/anhtv/ETC/Human_tracking/models/best.pt"

In [8]:
from ultralytics import YOLO

model = YOLO(MODEL_PATH)

# In thông tin tổng quan
print(model)

# In chi tiết kiến trúc
model.info()

# Số class
print("Number of classes:", model.model.nc)

# Tên class
print("Class names:", model.model.names)

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(96, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_s

In [9]:
print("Model stride:", model.model.stride)

Model stride: tensor([ 8., 16., 32.])


In [11]:
results = model.predict("/home/anhtv/ETC/Human_tracking/models/000001.jpg")

r = results[0]

print("Boxes shape:", r.boxes.xyxy.shape)
print("Conf:", r.boxes.conf)
print("Class:", r.boxes.cls)


image 1/1 /home/anhtv/ETC/Human_tracking/models/000001.jpg: 384x640 11 persons, 6 heads, 179.8ms
Speed: 5.8ms preprocess, 179.8ms inference, 9.2ms postprocess per image at shape (1, 3, 384, 640)
Boxes shape: torch.Size([17, 4])
Conf: tensor([0.9103, 0.8570, 0.8448, 0.8145, 0.7952, 0.7688, 0.7677, 0.6865, 0.5663, 0.4886, 0.4826, 0.4818, 0.4089, 0.3914, 0.3908, 0.3491, 0.2654])
Class: tensor([0., 0., 0., 1., 1., 0., 0., 0., 0., 1., 1., 0., 1., 0., 0., 0., 1.])


In [12]:
print(model.model.names)

{0: 'person', 1: 'head'}


In [13]:
from ultralytics import YOLO

model = YOLO("best.pt")
model.export(format="onnx", opset=12, nms=True)

Ultralytics 8.4.16 🚀 Python-3.12.3 torch-2.10.0+cu128 CPU (Intel Core i5-10400 2.90GHz)
Model summary (fused): 72 layers, 11,126,358 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from 'best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (85.4 MB)

ONNX: starting export with onnx 1.20.1 opset 12...
ONNX: slimming with onnxslim 0.1.88...
ONNX: export success ✅ 1.8s, saved as 'best.onnx' (42.7 MB)

Export complete (2.3s)
Results saved to /home/anhtv/ETC/Human_tracking/models
Predict:         yolo predict task=detect model=best.onnx imgsz=640 
Validate:        yolo val task=detect model=best.onnx imgsz=640 data=crowdhuman.yaml  
Visualize:       https://netron.app


'best.onnx'

In [14]:
import onnx

model = onnx.load("/home/anhtv/ETC/Human_tracking/models/best.onnx")

print("=== OUTPUT INFO ===")
for output in model.graph.output:
    print("Name:", output.name)
    print("Shape:", [
        dim.dim_value for dim in output.type.tensor_type.shape.dim
    ])
    print("Type:", output.type.tensor_type.elem_type)
    print("--------------------")

=== OUTPUT INFO ===
Name: output0
Shape: [1, 300, 6]
Type: 1
--------------------


In [16]:
import cv2
import numpy as np
import onnxruntime as ort

# Load model
session = ort.InferenceSession("/home/anhtv/ETC/Human_tracking/models/best.onnx")

input_name = session.get_inputs()[0].name
input_shape = session.get_inputs()[0].shape

# Load ảnh thật
img = cv2.imread("/work/models/000001.jpg")
img = cv2.resize(img, (640, 640))
img = img[:, :, ::-1]  # BGR → RGB
img = img.astype(np.float32) / 255.0
img = np.transpose(img, (2, 0, 1))  # HWC → CHW
img = np.expand_dims(img, 0)

# Inference
outputs = session.run(None, {input_name: img})

print("Output shape:", outputs[0].shape)
print(outputs[0][:5])

[ WARN:0@122.075] global loadsave.cpp:278 findDecoder imread_('/work/models/000001.jpg'): can't open/read file: check file path/integrity


error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'resize'


# Thông tin tiền xử lý

In [4]:
from ultralytics import YOLO
import torch

# Load model
model = YOLO("best.pt")

# Lấy số lớp (classes)
print("\n=== Number of classes ===")
print(model.model.nc)

# Lấy tên lớp
print("\n=== Class names ===")
print(model.model.names)

# Kiểm tra kích thước input mặc định
print("\n=== Model input size (stride based) ===")
print(model.model.stride)

# Kiểm tra first layer để biết số channel input
first_layer = list(model.model.model.children())[0]
print("\n=== First layer ===")
print(first_layer)

# Dummy forward để xem shape output
dummy = torch.zeros(1, 3, 640, 640)
output = model.model(dummy)
print("\n=== Output shape ===")
print(output[0].shape)


=== Number of classes ===
2

=== Class names ===
{0: 'person', 1: 'head'}

=== Model input size (stride based) ===
tensor([ 8., 16., 32.])

=== First layer ===
Conv(
  (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
  (act): SiLU(inplace=True)
)

=== Output shape ===
torch.Size([1, 6, 8400])


In [6]:
from ultralytics import YOLO

model = YOLO("best.pt")

print("===== MODEL ARGS =====")
for k, v in model.model.args.items():
    print(f"{k}: {v}")

===== MODEL ARGS =====
task: detect
data: crowdhuman.yaml
imgsz: 640
single_cls: False
model: best.pt


In [21]:
import torch
from ultralytics import YOLO

model = YOLO("best.pt")

dummy = torch.ones(1, 3, 640, 640) * 255

out1 = model.model(dummy)
out2 = model.model(dummy / 255)

print("Type out1:", type(out1))
print("Length out1:", len(out1))

# YOLOv8 thường trả tuple: (pred, aux)
pred1 = out1[0] if isinstance(out1, (list, tuple)) else out1
pred2 = out2[0] if isinstance(out2, (list, tuple)) else out2

print("Pred1 shape:", pred1.shape)
print("Pred2 shape:", pred2.shape)

print("Max out1:", pred1.abs().max())
print("Max out2:", pred2.abs().max())

first_layer = model.model.model[0]
print(first_layer)

Type out1: <class 'tuple'>
Length out1: 2
Pred1 shape: torch.Size([1, 6, 8400])
Pred2 shape: torch.Size([1, 6, 8400])
Max out1: tensor(636.0225)
Max out2: tensor(637.3620)
Conv(
  (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
  (act): SiLU(inplace=True)
)


In [24]:
import torch

first_layer = model.model.model[0]

def hook_fn(module, input, output):
    print("Layer1 max:", output.abs().max().item())

hook = first_layer.register_forward_hook(hook_fn)

dummy1 = torch.ones(1, 3, 640, 640) * 255
dummy2 = torch.ones(1, 3, 640, 640)

print("---- Input 255 ----")
model.model(dummy1)

print("---- Input 1 ----")
model.model(dummy2)

hook.remove()

---- Input 255 ----
Layer1 max: 11658.4248046875
---- Input 1 ----
Layer1 max: 47.37885665893555
